1.Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

pd.set_option('mode.chained_assignment', None)


2.Load Raw Dataset

In [2]:
raw_data_path = 'data/raw/dirty_cafe_sales.csv'
df = pd.read_csv(raw_data_path)

print("Shape:", df.shape)
display(df.head())
display(df.info())
display(df.describe())


Shape: (10000, 8)


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB


None

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
count,10000,9667,9862,9821,9827,7421,6735,9841
unique,10000,10,7,8,19,5,4,367
top,TXN_1961373,Juice,5,3.0,6.0,Digital Wallet,Takeaway,UNKNOWN
freq,1,1171,2013,2429,979,2291,3022,159


3.Missing Values Overview

In [3]:
missing_count = df.isnull().sum()
missing_percent = 100 * missing_count / len(df)
missing_overview = pd.DataFrame({
    'missing_count': missing_count,
    'missing_percent': missing_percent
}).sort_values(by='missing_percent', ascending=False)

os.makedirs('results/tables', exist_ok=True)
missing_overview.to_csv('results/tables/missing_overview_classification.csv', index=False)
display(missing_overview)


,missing_count,missing_percent
Location,3265,32.65
Payment Method,2579,25.79
Item,333,3.33
Price Per Unit,179,1.79
Total Spent,173,1.73
Transaction Date,159,1.59
Quantity,138,1.38
Transaction ID,0,0.00


4.Duplicates Check

In [4]:
# Check duplicates
dups = df.duplicated().sum()
print("Number of duplicate rows:", dups)

# Save duplicate rows to CSV
duplicate_rows = df[df.duplicated(keep=False)]
duplicate_rows.to_csv('results/tables/duplicate_rows_classification.csv', index=False)

# Drop duplicates
df = df.drop_duplicates()
print("Duplicates removed. New dataset shape:", df.shape)


Number of duplicate rows: 0
Duplicates removed. New dataset shape: (10000, 8)


5.Handle Missing Values

In [5]:
# Drop rows with missing target
df = df.dropna(subset=['Payment Method'])

# Numeric columns
numeric_cols = ['Quantity', 'Price Per Unit', 'Total Spent']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].fillna(df[col].median())

# Categorical columns
categorical_cols = ['Item', 'Location']
for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

# Transaction Date
df['Transaction Date'] = pd.to_datetime(df['Transaction Date'], errors='coerce')

display(df.head())


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4.0,1.0,8.0,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2.0,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11
